In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
import plotly.express as px

from pathlib import Path

In [ ]:
data_dir = Path("../../processed_data")
b19 = gpd.read_file(data_dir / "buildings" / "buildings_assigned_economic_activity.gpkg")
b23 = gpd.read_parquet(data_dir / "buildings" / "buildings_assigned_economic_activity.geoparquet")
admin = {n: gpd.read_file(data_dir / "boundaries" / "admin_boundaries.gpkg", layer=f"admin{n}") for n in (1, 2, 3)}
jic = pd.read_csv("jic_mapping.csv", skiprows=3)

In [ ]:
sectors = {
    1: "Primary",
    2: "Secondary",
    3: "Tertiary",
}

b3 = {}
for buildings, jic_year, gva_year in ((b19, 2005, 2019), (b23, 2016, 2023)):
    print(jic_year)

    gdp_cols = [c for c in buildings.columns if (c.endswith("_GDP") and c != "total_GDP")]
    three, letter = jic.loc[:, ["Three sector", f"Sector ({jic_year})"]].values.T
    letter = [f"{s.strip()}_GDP" for s in letter]
    jic_three = {l: t for l, t in zip(letter, three) if l in gdp_cols}

    cols = ["osm_id", "building_type", *jic_three.keys()]
    tmp = buildings.loc[:, cols].copy().dropna(subset=["osm_id"])
    if tmp["osm_id"].duplicated().any():
        agg = {"building_type": "first", **{c: "sum" for c in jic_three.keys()}}
        tmp = tmp.groupby("osm_id", as_index=False).agg(agg)
    tmp = tmp.set_index("osm_id")

    gdp_df = tmp.loc[:, jic_three.keys()].copy()
    gdp_df.columns = pd.MultiIndex.from_arrays([[jic_three[c] for c in gdp_df.columns], gdp_df.columns])
    coarse = gdp_df.T.groupby(level=0).sum().T

    # B USD / year
    print(coarse.sum() * 365 / 150 / 1e9)

    coarse["type"] = tmp["building_type"]
    b3[gva_year] = coarse

common_ids = b3[2019].index.intersection(b3[2023].index)
b3 = {year: df.loc[common_ids].sort_index() for year, df in b3.items()}

In [ ]:


sectors = {
    1: "Primary",
    2: "Secondary",
    3: "Tertiary",
}

# Choose coarse sector to inspect (1, 2, or 3).
for sector in sectors.keys():

    # Build dataframe aligned on osm_id index
    plot_df = pd.DataFrame({
        "gva_2019": b3[2019][sector],
        "gva_2023": b3[2023][sector],
        "building_type": b3[2019]["type"].fillna(b3[2023]["type"]).fillna("Unknown"),
        "jic2005_sector": b19.set_index("osm_id").loc[:, "sector_code"].fillna("Unknown"),
        "jic2016_sector": b23.set_index("osm_id").loc[:, "jic2016_sector"].fillna("Unknown"),
        "name": b23.set_index("osm_id").loc[:, "name"].fillna("Unknown"),
        "assigned_attribute": b23.set_index("osm_id").loc[:, "assigned_attribute"].fillna("Unknown"),
    }).dropna(subset=["gva_2019", "gva_2023"])

    # Expose osm_id in hover by promoting the index to a column.
    plot_df = plot_df.reset_index().rename(columns={"index": "osm_id"})

    # Log axes require strictly positive values.
    plot_df = plot_df.loc[(plot_df["gva_2019"] > 0) & (plot_df["gva_2023"] > 0)].copy()

    fig = px.scatter(
        plot_df,
        x="gva_2019",
        y="gva_2023",
        color="building_type",
        hover_data={
            "osm_id": True,
            "building_type": True,
            "gva_2019": ":.3g",
            "gva_2023": ":.3g",
            "jic2005_sector": True,
            "jic2016_sector": True,
            "name": True,
            "assigned_attribute": True,
        },
        labels={"gva_2019": "2019 GVA", "gva_2023": "2023 GVA"},
        title=sectors[sector],
        opacity=0.65,
    )

    lo = min(plot_df["gva_2019"].min(), plot_df["gva_2023"].min())
    hi = max(plot_df["gva_2019"].max(), plot_df["gva_2023"].max())
    log_range = [np.log10(lo), np.log10(hi)]
    fig.update_xaxes(type="log", range=log_range)
    fig.update_yaxes(type="log", range=log_range, scaleanchor="x", scaleratio=1)
    fig.update_layout(width=800, height=800)
    fig.update_xaxes(type="log")
    fig.update_yaxes(type="log")

    # Add 1:1 reference line over current data range.
    lo = min(plot_df["gva_2019"].min(), plot_df["gva_2023"].min())
    hi = max(plot_df["gva_2019"].max(), plot_df["gva_2023"].max())
    fig.add_shape(
        type="line",
        x0=lo,
        y0=lo,
        x1=hi,
        y1=hi,
        line={"dash": "dash", "color": "black", "width": 1},
    )

    fig.show()

In [ ]:

def plot_sectoral_GVA_admin_area(gva, buildings, admin, quantile_drop=0.05):
    sectors = (
        (1, "Primary", "YlGn"),
        (2, "Secondary", "YlOrBr"),
        (3, "Tertiary", "BuPu"),
    )

    # Spatial join buildings to desired admin level polygons
    df = gpd.GeoDataFrame(gva.loc[:, [1, 2, 3]].join(buildings.set_index("osm_id").loc[:, ["geometry"]]))
    df.geometry = df.geometry.representative_point()
    geom = admin.reset_index(names="admin_id").loc[:, ["admin_id", "geometry"]]
    df = df.sjoin(geom).loc[:, [1, 2, 3, "admin_id"]]

    # Sum by sector per admin polygon
    to_plot = geom.join(df.loc[:, [1, 2, 3, "admin_id"]].groupby("admin_id").sum()).to_crs(4326)

    f, axes = plt.subplots(3, 1, figsize=(8,11))
    cbar_formatter = ticker.ScalarFormatter()
    cbar_formatter.set_powerlimits((0, 0))
    for (sector, name, cmap_name), ax in zip(sectors, axes):
        cmap = plt.get_cmap(cmap_name)
        cmap.set_extremes(bad="pink", under="white", over="black")
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        to_plot.loc[:, [sector, "geometry"]].plot(
            sector,
            ax=ax,
            legend=True,
            cax=cax,
            cmap=cmap,
            legend_kwds={
                "extend": "both",
                "format": cbar_formatter,
            },
            vmin=to_plot.loc[:, [sector]].quantile(quantile_drop),
            vmax=to_plot.loc[:, [sector]].quantile(1 - quantile_drop),
        )
        ax.text(
            0.99,
            0.97,
            f"Total: {to_plot[sector].sum():1.1E} JMD d$^{{-1}}$",
            transform=ax.transAxes,
            horizontalalignment="right",
            verticalalignment="top",
            fontsize=8,
        )
        cax.set_ylabel(f"{name} GVA [JMD day$^{{-1}}$]")
        ax.grid(which="major", alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.1))


plot_sectoral_GVA_admin_area(b3[2023], b23, admin[1])
plot_sectoral_GVA_admin_area(b3[2023], b23, admin[2])
plot_sectoral_GVA_admin_area(b3[2023], b23, admin[3])